<a href="https://colab.research.google.com/github/cvilelahep/LIP_Internships26_SNDATLAS/blob/main/to_df.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
!pip install uproot
import uproot
import matplotlib.pyplot as plt
import pandas as pd
import awkward as ak

In [2]:
# Get the data
from google.colab import drive
drive.mount('/content/drive')
! mkdir -p /drive/MyDrive/LIP_internships26_SNDATLAS/
! wget -N https://cristova.web.cern.ch/cristova/share/LIP_Internships/epos_lhc_14TeV_v2.root -P /drive/MyDrive/LIP_internships26_SNDATLAS/
! wget -N https://cristova.web.cern.ch/cristova/share/LIP_Internships/pwgevents-PYTHIA-filtered_v2.root -P /drive/MyDrive/LIP_internships26_SNDATLAS/

! ls /drive/MyDrive/LIP_internships26_SNDATLAS/

data_path = "/drive/MyDrive/LIP_internships26_SNDATLAS/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--2026-07-13 09:09:53--  https://cristova.web.cern.ch/cristova/share/LIP_Internships/epos_lhc_14TeV_v2.root
Resolving cristova.web.cern.ch (cristova.web.cern.ch)... 137.138.55.232, 188.184.111.229, 188.184.96.45, ...
Connecting to cristova.web.cern.ch (cristova.web.cern.ch)|137.138.55.232|:443... connected.
HTTP request sent, awaiting response... 304 Not Modified
File ‘/drive/MyDrive/LIP_internships26_SNDATLAS/epos_lhc_14TeV_v2.root’ not modified on server. Omitting download.

--2026-07-13 09:09:54--  https://cristova.web.cern.ch/cristova/share/LIP_Internships/pwgevents-PYTHIA-filtered_v2.root
Resolving cristova.web.cern.ch (cristova.web.cern.ch)... 137.138.55.232, 188.184.111.229, 188.184.96.45, ...
Connecting to cristova.web.cern.ch (cristova.web.cern.ch)|137.138.55.232|:443... connected.
HTTP request sent, awaiting response... 304 Not Modified
File ‘/drive

In [3]:
signal = uproot.open(data_path+"/pwgevents-PYTHIA-filtered_v2.root:Events")
background = uproot.open(data_path+"/epos_lhc_14TeV_v2.root:Events")
signal.show()


name                 | typename                 | interpretation                
---------------------+--------------------------+-------------------------------
npdg                 | int32_t                  | AsDtype('>i4')
pdg                  | int64_t[]                | AsJagged(AsDtype('>i8'))
ncharge              | int32_t                  | AsDtype('>i4')
charge               | double[]                 | AsJagged(AsDtype('>f8'))
npx                  | int32_t                  | AsDtype('>i4')
px                   | double[]                 | AsJagged(AsDtype('>f8'))
npy                  | int32_t                  | AsDtype('>i4')
py                   | double[]                 | AsJagged(AsDtype('>f8'))
npz                  | int32_t                  | AsDtype('>i4')
pz                   | double[]                 | AsJagged(AsDtype('>f8'))
neta                 | int32_t                  | AsDtype('>i4')
eta                  | double[]                 | AsJagged(AsDtype('>f8')

In [6]:
def makeDataFrame(root_tree):
  this_df = pd.DataFrame()
  this_df["n"] = 0.0

  n = 0
  for i_event, event in enumerate(root_tree["npt"].array()):
    if i_event % 1000 == 0:
      print(f"Processing event {i_event}")
    this_df._set_value(index=n, col="n", value = event)
    n+=1

  for key in root_tree.keys():
    print(f"Processing {key}")
    if key.startswith("n") or key.startswith("m") or key.startswith("charge") or key.startswith("pdg"): continue

    for i in range(1, 4):
      this_df[key + str(i)] = 0.0
    n = 0
    for event in root_tree[key].array():
      for j in range(0, min(3,len(event))):
        this_df._set_value(index=n, col=key + str(j+1), value = event[j])
      n += 1
  #this_df._set_value(index=n, col=key + str(i), value = event[j])

  #this_df["mpt"] = root_tree["mpt"].array()
  this_df["nu_eta"] = root_tree["nu_eta"].array()
  this_df["nu_phi"] = root_tree["nu_phi"].array()
  this_df["nu_E"] = root_tree["nu_E"].array()


  this_df["dphi2"] = this_df["phi2"] - this_df["phi1"]
  this_df["dphi3"] = this_df["phi3"] - this_df["phi1"]
  this_df["p1"] = np.sqrt(this_df["px1"]**2 + this_df["py1"]**2 + this_df["pz1"]**2)
  this_df["p2"] = np.sqrt(this_df["px2"]**2 + this_df["py2"]**2 + this_df["pz2"]**2)
  this_df["p3"] = np.sqrt(this_df["px3"]**2 + this_df["py3"]**2 + this_df["pz3"]**2)
  this_df["nu_dphi1"] = this_df["phi1"] - this_df["nu_phi"]
  this_df["nu_dphi2"] = this_df["phi2"] - this_df["nu_phi"]
  this_df["nu_dphi3"] = this_df["phi3"] - this_df["nu_phi"]


  this_df = this_df.fillna(0)
  this_df.head()
  return this_df

In [ ]:
background_df = makeDataFrame(signal)
signal_df = makeDataFrame(background)

Processing npdg
Processing pdg
Processing ncharge
Processing charge
Processing npx
Processing px


In [ ]:
signal_df.tail()

In [ ]:
signal_df["target"] = 1.
background_df["target"] = 0.
signal_df = signal_df.drop(columns=["phi1", "phi2", "phi3"])
background_df = background_df.drop(columns=["phi1", "phi2", "phi3"])


In [ ]:
signal_df.to_csv("Data/signal_v2_eta_E.csv", index=False)
background_df.to_csv("Data/background_v2_eta_E.csv", index=False)

In [ ]:

plt.figure()

plt.hist(
    signal_df["p2"][signal_df["p2"] != 0].dropna(),
    bins=80,
    range=(0, 50),
    histtype="step",
    linewidth=1,
    color="b",
    label="Signal"
)

plt.hist(
    background_df["p2"][background_df["p2"] != 0].dropna(),
    bins=80,
    range=(0, 50),
    histtype="step",
    linewidth=1,
    color="r",
    label="Background"
)

plt.xlabel("p2")
plt.legend()
plt.show()